# 77 - ColPali 遅延相互作用モデル

## 概要
本ノートブックでは、ColQwen2（ColPali アーキテクチャ）による **遅延相互作用（Late Interaction）** 検索を実験する。
これまでの Notebook 61-76 で扱った対照学習モデル（SigLIP 2, CLIP-L）は画像・テキストを**単一ベクトル**に圧縮していたが、
ColPali は根本的に異なるパラダイムを用いる。

## 対照学習の単一ベクトル vs 遅延相互作用のマルチベクトル

### 単一ベクトル（Single-Vector）方式
- 画像全体 → 1つのベクトル（例: SigLIP 2 Large = 1024次元）
- クエリテキスト全体 → 1つのベクトル
- スコアリング: `cosine_similarity(image_vec, query_vec)` — O(1)
- **利点**: 高速、省ストレージ、ANN インデックス（HNSW）が使える
- **欠点**: 画像の局所的な特徴が圧縮により失われる

### マルチベクトル（Late Interaction）方式
- 画像 → N個のパッチベクトル（例: 1024パッチ x 128次元）
- クエリ → M個のトークンベクトル（例: 32トークン x 128次元）
- スコアリング: **MaxSim** — 各クエリトークンについて最も類似するパッチを見つけ、その最大類似度を合計
  - `score = sum(max(sim(q_i, p_j) for all patches j) for all query tokens i)`
- **利点**: きめ細かいマッチング、局所特徴の保持、注意の可視化が可能
- **欠点**: ストレージ大、スコアリングが遅い（O(N*M)）、ANN が使いにくい

## ColQwen2
- **ベースモデル**: Qwen2-VL-2B（視覚言語モデル）
- **学習**: ドキュメント検索タスクで fine-tuned
- **出力**: 各画像パッチに対応する 128 次元の埋め込みベクトル群
- **VRAM**: ~8GB（fp16）

## 評価
- MaxSim スコアリングによるテキスト→画像検索
- 同じ 18 テストクエリ（Notebook 62/76 と共通）で P@K, MRR を計算
- SigLIP 2 Large 単一ベクトル方式との品質・速度・ストレージ比較
- 注意（Attention）ヒートマップの可視化

In [ ]:
import json
import time
from datetime import datetime
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.notebook import tqdm

In [ ]:
# ColPali/ColQwen2 のインポート
from colpali_engine.models import ColQwen2, ColQwen2Processor

print("colpali_engine imported successfully.")

## 1. 設定とデータ読み込み

In [ ]:
DB_PATH = Path("../data/images.duckdb")
COLQWEN_MODEL = "vidore/colqwen2-v1.0"
SIGLIP2_MODEL = "google/siglip2-large-patch16-256"

# 画像カタログの読み込み
conn = duckdb.connect(str(DB_PATH), read_only=True)

catalog_df = conn.execute("""
    SELECT id, file_path, category, file_name
    FROM image_catalog
    ORDER BY id
""").fetchdf()

# カテゴリリスト（検索評価用）
categories = catalog_df['category'].tolist()

conn.close()

print(f"Total images: {len(catalog_df)}")
print(f"\nCategories:")
for cat, count in catalog_df['category'].value_counts().items():
    print(f"  {cat}: {count}")

## 2. ColQwen2 モデルの読み込み

In [ ]:
print(f"Loading ColQwen2 model: {COLQWEN_MODEL}")
print("This may take several minutes for the first download (~8GB)...")

model = ColQwen2.from_pretrained(
    COLQWEN_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
)
processor = ColQwen2Processor.from_pretrained(COLQWEN_MODEL)

model.eval()

print(f"Model loaded.")
if torch.cuda.is_available():
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## 3. 遅延相互作用の概念説明

### 単一ベクトル（対照学習）
```
画像 ──→ ViT ──→ Pooling ──→ [1 x 1024]  (1つのベクトル)
テキスト ──→ Tokenizer ──→ Transformer ──→ Pooling ──→ [1 x 1024]

Score = cosine_sim([1 x 1024], [1 x 1024]) = スカラー値
```

### マルチベクトル（遅延相互作用 / Late Interaction）
```
画像 ──→ ViT ──→ Projection ──→ [N x 128]  (Nパッチのベクトル群)
テキスト ──→ Tokenizer ──→ Transformer ──→ Projection ──→ [M x 128]  (Mトークンのベクトル群)

Score = MaxSim([M x 128], [N x 128])
     = Σ_i max_j sim(q_i, p_j)
     各クエリトークン i について、最も類似するパッチ j の類似度を合計
```

### MaxSim の直感的理解
- クエリ「公園の桜の木」→ トークン [「公園」, 「の」, 「桜」, 「の」, 「木」]
- 各トークンが画像のどのパッチと最もマッチするかを個別に計算
- 「桜」トークンは桜が写っているパッチと高い類似度
- 「公園」トークンは公園の風景パッチと高い類似度
- → 画像の**異なる部分**がクエリの**異なる部分**と個別にマッチできる

## 4. 全画像のマルチベクトル表現を生成

In [ ]:
# 全画像のマルチベクトル埋め込みを生成
BATCH_SIZE = 4  # ColQwen2 はメモリを多く使うため小さめのバッチ

all_image_embeddings = []  # List of tensors, each [N_patches, 128]

print(f"Generating multi-vector embeddings for {len(catalog_df)} images...")
print(f"Batch size: {BATCH_SIZE}")

start_time = time.time()

for batch_start in tqdm(range(0, len(catalog_df), BATCH_SIZE), desc="Image Embedding"):
    batch_end = min(batch_start + BATCH_SIZE, len(catalog_df))
    batch_rows = catalog_df.iloc[batch_start:batch_end]
    
    # 画像を読み込み
    batch_images = []
    for _, row in batch_rows.iterrows():
        img = Image.open(row['file_path']).convert('RGB')
        batch_images.append(img)
    
    # ColQwen2 で処理
    batch_inputs = processor.process_images(batch_images).to(model.device)
    
    with torch.no_grad():
        batch_embeddings = model(**batch_inputs)
    
    # 各画像の埋め込みを保存（CPU に移動してメモリ節約）
    for emb in batch_embeddings:
        all_image_embeddings.append(emb.cpu().float())

embed_time = time.time() - start_time

print(f"\nEmbedding completed in {embed_time:.1f}s")
print(f"Speed: {len(catalog_df) / embed_time:.2f} images/sec")
print(f"\nEmbedding shapes (first 5 images):")
for i in range(min(5, len(all_image_embeddings))):
    print(f"  Image {i}: {all_image_embeddings[i].shape}")

# パッチ数の統計
patch_counts = [emb.shape[0] for emb in all_image_embeddings]
print(f"\nPatch count statistics:")
print(f"  Mean:  {np.mean(patch_counts):.0f}")
print(f"  Min:   {np.min(patch_counts)}")
print(f"  Max:   {np.max(patch_counts)}")
print(f"  Embed dim: {all_image_embeddings[0].shape[1]}")

## 5. MaxSim スコアリング関数の定義

In [ ]:
def compute_maxsim_score(query_embedding: torch.Tensor, image_embedding: torch.Tensor) -> float:
    """
    MaxSim スコアを計算する。
    
    Args:
        query_embedding: [M, D] — クエリのトークン埋め込み
        image_embedding: [N, D] — 画像のパッチ埋め込み
    
    Returns:
        MaxSim スコア（各クエリトークンの最大類似度の合計）
    """
    # コサイン類似度行列: [M, N]
    q_norm = query_embedding / query_embedding.norm(dim=1, keepdim=True).clamp(min=1e-8)
    p_norm = image_embedding / image_embedding.norm(dim=1, keepdim=True).clamp(min=1e-8)
    sim_matrix = torch.mm(q_norm, p_norm.T)  # [M, N]
    
    # 各クエリトークンについて最大類似度を取得し合計
    max_sim_per_token = sim_matrix.max(dim=1).values  # [M]
    score = max_sim_per_token.sum().item()
    
    return score


def compute_maxsim_matrix(query_embedding: torch.Tensor, image_embedding: torch.Tensor) -> torch.Tensor:
    """
    MaxSim のための類似度行列を返す（可視化用）。
    
    Returns:
        sim_matrix: [M, N] — 各クエリトークンと各画像パッチの類似度
    """
    q_norm = query_embedding / query_embedding.norm(dim=1, keepdim=True).clamp(min=1e-8)
    p_norm = image_embedding / image_embedding.norm(dim=1, keepdim=True).clamp(min=1e-8)
    return torch.mm(q_norm, p_norm.T)


# テスト
test_query = "a park with trees"
batch_queries = processor.process_queries([test_query]).to(model.device)
with torch.no_grad():
    test_query_emb = model(**batch_queries)[0].cpu().float()

test_score = compute_maxsim_score(test_query_emb, all_image_embeddings[0])
print(f"Test query: '{test_query}'")
print(f"Query embedding shape: {test_query_emb.shape}")
print(f"MaxSim score (vs image 0): {test_score:.4f}")

## 6. 18 標準テストクエリの定義と検索実行

In [ ]:
# テストクエリの定義（Notebook 62/76 と同じ）
test_queries = [
    # カテゴリ固有のクエリ（英語）
    {"query": "a park with trees and nature", "expected_categories": ["KashiwaVillagePark2026"], "type": "specific", "lang": "en"},
    {"query": "village park with greenery", "expected_categories": ["KashiwaVillagePark2026"], "type": "specific", "lang": "en"},
    {"query": "night cityscape Tokyo", "expected_categories": ["TokyoNight202505"], "type": "specific", "lang": "en"},
    {"query": "city lights at night", "expected_categories": ["TokyoNight202505"], "type": "specific", "lang": "en"},
    {"query": "Python conference presentation", "expected_categories": ["EuroPython2025", "PyConJP2025", "PyConJP2025-PreCampHiroshima"], "type": "specific", "lang": "en"},
    {"query": "conference hall with audience", "expected_categories": ["EuroPython2025", "PyConJP2025", "PyConJP2025-PreCampHiroshima"], "type": "specific", "lang": "en"},
    {"query": "speaker giving a talk", "expected_categories": ["EuroPython2025", "PyConJP2025", "PyConJP2025-PreCampHiroshima"], "type": "specific", "lang": "en"},
    {"query": "portrait photo of a person", "expected_categories": ["terada"], "type": "specific", "lang": "en"},
    # 一般的なクエリ（英語）
    {"query": "outdoor nature scene", "expected_categories": ["KashiwaVillagePark2026"], "type": "general", "lang": "en"},
    {"query": "urban night photography", "expected_categories": ["TokyoNight202505"], "type": "general", "lang": "en"},
    {"query": "tech conference event", "expected_categories": ["EuroPython2025", "PyConJP2025", "PyConJP2025-PreCampHiroshima"], "type": "general", "lang": "en"},
    {"query": "people at an event", "expected_categories": ["EuroPython2025", "PyConJP2025", "PyConJP2025-PreCampHiroshima"], "type": "general", "lang": "en"},
    # 日本語クエリ
    {"query": "公園の自然風景", "expected_categories": ["KashiwaVillagePark2026"], "type": "specific", "lang": "ja"},
    {"query": "東京の夜景", "expected_categories": ["TokyoNight202505"], "type": "specific", "lang": "ja"},
    {"query": "カンファレンスの講演", "expected_categories": ["EuroPython2025", "PyConJP2025", "PyConJP2025-PreCampHiroshima"], "type": "specific", "lang": "ja"},
    {"query": "人物の写真", "expected_categories": ["terada"], "type": "specific", "lang": "ja"},
    {"query": "緑の木々", "expected_categories": ["KashiwaVillagePark2026"], "type": "general", "lang": "ja"},
    {"query": "夜の街", "expected_categories": ["TokyoNight202505"], "type": "general", "lang": "ja"},
]

print(f"Total test queries: {len(test_queries)}")
print(f"  English: {sum(1 for q in test_queries if q['lang'] == 'en')}")
print(f"  Japanese: {sum(1 for q in test_queries if q['lang'] == 'ja')}")

In [ ]:
# 全クエリのマルチベクトル埋め込みを生成
print("Generating query embeddings...")

query_embeddings = []
for q_info in tqdm(test_queries, desc="Query Embedding"):
    batch = processor.process_queries([q_info["query"]]).to(model.device)
    with torch.no_grad():
        q_emb = model(**batch)[0].cpu().float()
    query_embeddings.append(q_emb)

print(f"Generated {len(query_embeddings)} query embeddings.")
print(f"Query embedding shapes (first 5):")
for i in range(min(5, len(query_embeddings))):
    print(f"  '{test_queries[i]['query']}': {query_embeddings[i].shape}")

In [ ]:
# MaxSim スコアリングで全クエリを検索
K_VALUES = [1, 5, 10, 20]
MAX_K = max(K_VALUES)

print(f"Running MaxSim search for {len(test_queries)} queries x {len(catalog_df)} images...")

search_start = time.time()
search_results = []

for q_idx, q_info in enumerate(tqdm(test_queries, desc="MaxSim Search")):
    q_emb = query_embeddings[q_idx]
    
    # 全画像に対するスコアを計算
    scores = []
    for img_emb in all_image_embeddings:
        score = compute_maxsim_score(q_emb, img_emb)
        scores.append(score)
    
    scores = np.array(scores)
    sorted_indices = np.argsort(scores)[::-1]  # 降順ソート
    
    # 上位K件の結果
    top_k_indices = sorted_indices[:MAX_K]
    top_k_scores = scores[top_k_indices]
    top_k_categories = [categories[idx] for idx in top_k_indices]
    
    expected_cats = q_info["expected_categories"]
    is_relevant = [cat in expected_cats for cat in top_k_categories]
    
    # Precision@K
    precision_at_k = {}
    for k in K_VALUES:
        relevant_in_k = sum(is_relevant[:k])
        precision_at_k[f"P@{k}"] = relevant_in_k / k
    
    # MRR
    mrr = 0.0
    for i, rel in enumerate(is_relevant):
        if rel:
            mrr = 1.0 / (i + 1)
            break
    
    # Hit Rate@K
    hit_rate_at_k = {}
    for k in K_VALUES:
        hit_rate_at_k[f"HR@{k}"] = 1.0 if any(is_relevant[:k]) else 0.0
    
    result = {
        "query": q_info["query"],
        "lang": q_info["lang"],
        "type": q_info["type"],
        "expected_categories": expected_cats,
        "top_k_indices": top_k_indices.tolist(),
        "top_k_scores": top_k_scores.tolist(),
        "top_k_categories": top_k_categories,
        "MRR": mrr,
        **precision_at_k,
        **hit_rate_at_k,
    }
    search_results.append(result)
    print(f"  {q_info['query']:40s} | P@1={precision_at_k['P@1']:.2f} | P@10={precision_at_k['P@10']:.2f} | MRR={mrr:.3f}")

search_time = time.time() - search_start
print(f"\nSearch completed in {search_time:.1f}s")
print(f"Average query time: {search_time / len(test_queries):.2f}s per query")

## 7. P@K, MRR 評価

In [ ]:
# 結果を DataFrame に変換
results_df = pd.DataFrame(search_results)

# 全体メトリクス
metrics = ['P@1', 'P@5', 'P@10', 'P@20', 'MRR', 'HR@1', 'HR@5', 'HR@10']
overall_means = results_df[metrics].mean()

print("=" * 60)
print("Overall Metrics (ColQwen2 Late Interaction)")
print("=" * 60)
for metric in metrics:
    print(f"  {metric:8s}: {overall_means[metric]:.4f}")

In [ ]:
# 言語別メトリクス
print("\n" + "=" * 60)
print("Metrics by Language")
print("=" * 60)

lang_results = results_df.groupby('lang')[metrics].mean()
display(lang_results.round(4))

In [ ]:
# クエリタイプ別メトリクス
print("\n" + "=" * 60)
print("Metrics by Query Type")
print("=" * 60)

type_results = results_df.groupby('type')[metrics].mean()
display(type_results.round(4))

In [ ]:
# 全クエリ別の詳細結果
print("\n" + "=" * 80)
print("Per-Query Results")
print("=" * 80)

display_df = results_df[['query', 'lang', 'type', 'P@1', 'P@5', 'P@10', 'MRR']].copy()
display_df.columns = ['Query', 'Lang', 'Type', 'P@1', 'P@5', 'P@10', 'MRR']
display(display_df.round(4))

## 8. SigLIP 2 Large 単一ベクトル方式との比較

In [ ]:
# SigLIP 2 Large の結果を読み込み
eval_dir = Path("../data/evaluations")

# SigLIP 2 Large text search 結果
siglip2_files = sorted(eval_dir.glob("text_search_google_siglip2*large*.json")) + \
                sorted(eval_dir.glob("siglip2_large_onnx*.json"))

siglip2_data = None
if siglip2_files:
    with open(siglip2_files[-1]) as f:
        siglip2_data = json.load(f)
    print(f"Loaded SigLIP 2 Large results: {siglip2_files[-1].name}")

# もしテキスト検索結果が見つからない場合、SigLIP 2 評価結果を探す
if siglip2_data is None:
    siglip2_eval_files = sorted(eval_dir.glob("google_siglip2*large*.json"))
    if siglip2_eval_files:
        with open(siglip2_eval_files[-1]) as f:
            siglip2_data = json.load(f)
        print(f"Loaded SigLIP 2 evaluation: {siglip2_eval_files[-1].name}")

# 比較テーブル
print("\n" + "=" * 80)
print("Comparison: Single-Vector (SigLIP 2 Large) vs Late Interaction (ColQwen2)")
print("=" * 80)

colpali_overall_mrr = overall_means['MRR']
colpali_en_mrr = lang_results.loc['en', 'MRR'] if 'en' in lang_results.index else 0
colpali_ja_mrr = lang_results.loc['ja', 'MRR'] if 'ja' in lang_results.index else 0

print(f"\n{'Metric':<35} {'SigLIP 2 Large':>15} {'ColQwen2':>15}")
print("-" * 67)
print(f"{'Approach':<35} {'Single-Vector':>15} {'Late Interaction':>15}")
print(f"{'Embedding dim':<35} {'1024':>15} {'128 x N patches':>15}")

# SigLIP 2 結果があれば比較
if siglip2_data:
    if 'overall_metrics' in siglip2_data:
        s2_mrr = siglip2_data['overall_metrics'].get('MRR', 0)
    elif 'by_language' in siglip2_data:
        s2_mrr = 0  # 個別に計算
    else:
        s2_mrr = 0
    print(f"{'Overall MRR':<35} {s2_mrr:>15.4f} {colpali_overall_mrr:>15.4f}")

print(f"{'ColQwen2 MRR (EN)':<35} {'—':>15} {colpali_en_mrr:>15.4f}")
print(f"{'ColQwen2 MRR (JA)':<35} {'—':>15} {colpali_ja_mrr:>15.4f}")
print(f"{'VRAM':<35} {'~3 GB':>15} {'~8 GB':>15}")
print(f"{'Embed speed':<35} {'~10 img/s':>15} {len(catalog_df)/embed_time:>12.1f} img/s")
print(f"{'Search time':<35} {'< 1ms/query':>15} {search_time/len(test_queries):>11.2f}s/query")

## 9. Attention ヒートマップの可視化

遅延相互作用の最大の利点の一つは、**どの画像パッチがどのクエリトークンとマッチしたか**を可視化できること。
各クエリトークンの MaxSim 先パッチを画像上にヒートマップとして重畳表示する。

In [ ]:
def visualize_attention_heatmap(query: str, query_emb: torch.Tensor,
                                image_emb: torch.Tensor, image_path: str,
                                processor_obj, ax=None):
    """
    クエリと画像の MaxSim 注意マップを可視化する。
    
    各パッチについて、全クエリトークンからの最大類似度を集約してヒートマップを生成する。
    """
    # 類似度行列: [M_tokens, N_patches]
    sim_matrix = compute_maxsim_matrix(query_emb, image_emb)
    
    # 各パッチに対する最大類似度（全クエリトークンの中で）
    # → どのパッチが最も「重要」かを示す
    patch_importance = sim_matrix.max(dim=0).values.numpy()  # [N_patches]
    
    # パッチ数から正方形のグリッドサイズを推定
    n_patches = len(patch_importance)
    grid_size = int(np.ceil(np.sqrt(n_patches)))
    
    # ヒートマップ用にパディング
    heatmap = np.zeros(grid_size * grid_size)
    heatmap[:n_patches] = patch_importance
    heatmap = heatmap.reshape(grid_size, grid_size)
    
    # 画像を読み込み
    img = Image.open(image_path).convert('RGB')
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 8))
    
    ax.imshow(img)
    
    # ヒートマップを画像サイズにリサイズしてオーバーレイ
    from PIL import Image as PILImage
    heatmap_resized = np.array(
        PILImage.fromarray(
            ((heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8) * 255).astype(np.uint8)
        ).resize((img.width, img.height), PILImage.BILINEAR)
    ) / 255.0
    
    ax.imshow(heatmap_resized, cmap='jet', alpha=0.4)
    ax.set_title(f'Query: "{query}"', fontsize=10, wrap=True)
    ax.axis('off')
    
    return sim_matrix


print("Attention visualization function defined.")

In [ ]:
# 3-4 サンプルクエリでヒートマップを表示
visualization_queries = [
    {"query": "a park with trees and nature", "idx": 0},   # 英語・公園
    {"query": "city lights at night", "idx": 3},            # 英語・夜景
    {"query": "speaker giving a talk", "idx": 6},           # 英語・カンファレンス
    {"query": "公園の自然風景", "idx": 12},                   # 日本語・公園
]

for viz_info in visualization_queries:
    q_idx = viz_info["idx"]
    query_text = viz_info["query"]
    q_emb = query_embeddings[q_idx]
    
    # このクエリの Top-3 検索結果に対してヒートマップを表示
    top3_indices = search_results[q_idx]["top_k_indices"][:3]
    top3_scores = search_results[q_idx]["top_k_scores"][:3]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    for i, (img_idx, score) in enumerate(zip(top3_indices, top3_scores)):
        row = catalog_df.iloc[img_idx]
        img_emb = all_image_embeddings[img_idx]
        
        visualize_attention_heatmap(
            query_text, q_emb, img_emb,
            row['file_path'], processor, ax=axes[i]
        )
        axes[i].set_title(
            f'Rank {i+1} (score={score:.1f})\n[{row["category"]}] {row["file_name"][:30]}',
            fontsize=9
        )
    
    plt.suptitle(f'MaxSim Attention: "{query_text}"', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print()

In [ ]:
# トークンレベルの詳細可視化（1つのクエリ・画像ペアで）
detail_q_idx = 0  # "a park with trees and nature"
detail_img_idx = search_results[detail_q_idx]["top_k_indices"][0]  # Top-1 画像

q_emb = query_embeddings[detail_q_idx]
img_emb = all_image_embeddings[detail_img_idx]
sim_matrix = compute_maxsim_matrix(q_emb, img_emb)  # [M, N]

# 各トークンの最大類似度とその位置
max_sim_values, max_sim_indices = sim_matrix.max(dim=1)  # [M]

print(f"Query: '{test_queries[detail_q_idx]['query']}'")
print(f"Image: {catalog_df.iloc[detail_img_idx]['file_name']}")
print(f"Query tokens: {q_emb.shape[0]}, Image patches: {img_emb.shape[0]}")
print(f"\nToken-level MaxSim:")
print(f"  Max similarity per token (top 10):")
sorted_token_sims = max_sim_values.sort(descending=True)
for i in range(min(10, len(sorted_token_sims.values))):
    token_idx = sorted_token_sims.indices[i].item()
    sim_val = sorted_token_sims.values[i].item()
    patch_idx = max_sim_indices[token_idx].item()
    print(f"    Token {token_idx:3d} → Patch {patch_idx:4d}: sim = {sim_val:.4f}")

## 10. ストレージ分析

マルチベクトル方式と単一ベクトル方式のストレージ要件を比較する。

In [ ]:
# ストレージ分析
n_images = len(catalog_df)
embed_dim_colpali = all_image_embeddings[0].shape[1]  # 128
total_patches = sum(emb.shape[0] for emb in all_image_embeddings)
mean_patches = total_patches / n_images

# 単一ベクトル（SigLIP 2 Large: 1024 dim, float32）
single_vector_bytes = n_images * 1024 * 4  # float32
single_vector_mb = single_vector_bytes / (1024 ** 2)

# マルチベクトル（ColQwen2: N patches x 128 dim, float32）
multi_vector_bytes = total_patches * embed_dim_colpali * 4  # float32
multi_vector_mb = multi_vector_bytes / (1024 ** 2)

# マルチベクトル（float16 で保存した場合）
multi_vector_fp16_bytes = total_patches * embed_dim_colpali * 2
multi_vector_fp16_mb = multi_vector_fp16_bytes / (1024 ** 2)

print("=" * 60)
print("Storage Analysis")
print("=" * 60)

print(f"\n--- Single-Vector (SigLIP 2 Large) ---")
print(f"  Dimensions:      1024")
print(f"  Vectors/image:   1")
print(f"  Total vectors:   {n_images:,}")
print(f"  Storage (fp32):  {single_vector_mb:.2f} MB")
print(f"  Per image:       {1024 * 4 / 1024:.1f} KB")

print(f"\n--- Multi-Vector (ColQwen2) ---")
print(f"  Dimensions:      {embed_dim_colpali}")
print(f"  Mean patches:    {mean_patches:.0f}")
print(f"  Total vectors:   {total_patches:,}")
print(f"  Storage (fp32):  {multi_vector_mb:.2f} MB")
print(f"  Storage (fp16):  {multi_vector_fp16_mb:.2f} MB")
print(f"  Per image:       {multi_vector_bytes / n_images / 1024:.1f} KB (fp32)")

print(f"\n--- Comparison ---")
print(f"  Storage ratio (multi/single):  {multi_vector_mb / single_vector_mb:.1f}x (fp32)")
print(f"  Storage ratio (multi fp16/single fp32):  {multi_vector_fp16_mb / single_vector_mb:.1f}x")

## 11. 結果の保存

In [ ]:
output_data = {
    "notebook": "77-colpali-late-interaction",
    "model_name": COLQWEN_MODEL,
    "timestamp": datetime.now().isoformat(),
    "total_images": len(catalog_df),
    "total_queries": len(test_queries),
    "approach": "late_interaction",
    "embedding_info": {
        "embed_dim": embed_dim_colpali,
        "mean_patches_per_image": float(mean_patches),
        "min_patches": int(np.min(patch_counts)),
        "max_patches": int(np.max(patch_counts)),
        "total_patch_vectors": int(total_patches),
    },
    "timing": {
        "embedding_time_seconds": embed_time,
        "embedding_speed_img_per_sec": len(catalog_df) / embed_time,
        "search_time_seconds": search_time,
        "search_time_per_query": search_time / len(test_queries),
    },
    "overall_metrics": {metric: float(overall_means[metric]) for metric in metrics},
    "by_language": {
        lang: {metric: float(lang_results.loc[lang, metric]) for metric in metrics}
        for lang in lang_results.index
    },
    "by_type": {
        qtype: {metric: float(type_results.loc[qtype, metric]) for metric in metrics}
        for qtype in type_results.index
    },
    "per_query_results": [
        {
            "query": r["query"],
            "lang": r["lang"],
            "type": r["type"],
            "MRR": r["MRR"],
            "P@1": r["P@1"],
            "P@5": r["P@5"],
            "P@10": r["P@10"],
        }
        for r in search_results
    ],
    "storage_analysis": {
        "single_vector_mb": single_vector_mb,
        "multi_vector_fp32_mb": multi_vector_mb,
        "multi_vector_fp16_mb": multi_vector_fp16_mb,
        "storage_ratio_fp32": multi_vector_mb / single_vector_mb,
    },
}

output_path = Path("../data/evaluations") / f"77_colpali_late_interaction_{datetime.now().strftime('%Y-%m-%d')}.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w") as f:
    json.dump(output_data, f, indent=2, ensure_ascii=False)

print(f"Results saved to: {output_path}")

## 12. GPU メモリのクリーンアップ

In [ ]:
del model, processor
del all_image_embeddings, query_embeddings

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU memory cleared.")
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## まとめ・評価・考察

### 実行状況: SKIP（colpali-engine が transformers 5.x と非互換）

`colpali-engine>=0.3.0` は `transformers<4.58.0` を要求するが、本プロジェクトは `transformers>=5.0.0` を使用しているため、依存関係が解決できず実行できなかった。

**技術的背景:**
- `colpali-engine` の `ColQwen2` は transformers の内部 API に強く依存しており、5.x の破壊的変更に対応していない。
- `uv sync` で他のパッケージ（Qwen2.5-VL 用の `qwen-vl-utils` 等）と共存させることができない。
- 別の依存グループ (`[dependency-groups]`) に分離しても、`uv` はデフォルトで全グループを解決するため回避不可。

**対処方針:**
- `colpali-engine` が transformers 5.x に対応したバージョンをリリースするまで待機。
- または `transformers<5.0.0` の別仮想環境を構築して単独実行。

### ColPali / 遅延相互作用の概要（参考情報）

| 項目 | 単一ベクトル (SigLIP 2) | マルチベクトル (ColQwen2) |
|------|:---:|:---:|
| 表現方式 | 画像→1ベクトル (1024d) | 画像→Nパッチベクトル (128d×N) |
| スコアリング | cosine_sim: O(1) | MaxSim: O(N×M) |
| ANNインデックス | HNSW使用可 | 困難 |
| ストレージ | ~4KB/画像 | ~数十KB/画像 |
| 局所特徴の保持 | 圧縮により失われる | 保持される |
| 可視化 | 不可 | Attention ヒートマップ可能 |

**遅延相互作用の利点:**
- 各クエリトークンが画像の異なるパッチと個別にマッチできるため、局所的な特徴（特定の物体、テキスト等）に対する検索精度が向上する可能性がある。
- 「なぜこの画像が検索されたか」をトークン×パッチの類似度行列で説明できる。

**実用的なトレードオフ:**
- ストレージと検索速度のコストが単一ベクトル方式に比べ大幅に増加。
- 大規模データセットでは、単一ベクトル（HNSW）で粗い検索→マルチベクトルでリランキングのハイブリッドが現実的。
- 本プロジェクトの378画像規模では実用性は高いが、pyconjp 規模（23,628画像）以上ではストレージ・速度が課題となる。